# Tìm kiếm điều luật cho câu hỏi gõ thiếu dấu

Nhóm 08, môn Xử lý ngôn ngữ tự nhiên.


## 1. Môi trường

In [11]:
import glob, os, subprocess, sys, time
from pathlib import Path

TREN_KAGGLE = Path("/kaggle/input").exists()
if TREN_KAGGLE:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "bm25s", "pyvi"], check=True)
    tim = glob.glob("/kaggle/input/**/src/vlr/config.py", recursive=True)
    assert tim, "Chưa thêm dataset vlr-demo vào notebook"
    GOC = Path(tim[0]).parents[2]
else:
    GOC = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    os.environ["HF_HUB_OFFLINE"] = "1"   # máy nhóm đã có mô hình trong cache
sys.path.insert(0, str(GOC / "src"))

import pandas as pd
import torch
pd.set_option("display.max_colwidth", 80)
print("Thư mục dữ liệu:", GOC)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "không có, chạy trên CPU")

Thư mục dữ liệu: /kaggle/input/datasets/messiac/vlr-demo
GPU: Tesla T4


## 2. Nạp mô hình

In [12]:
from vlr import config, fusion, textnorm, tra_cuu

t0 = time.time()
tro_ly = tra_cuu.TroLyTraCuu()
tro_ly.de.encode_queries(["khởi động"])   # nạp mô hình ngữ nghĩa ngay, lúc demo khỏi chờ
K = config.TOPK_FUSION
qs = pd.read_parquet(config.QUERIES_PATH).set_index("query_id")["text"]
qr = pd.read_parquet(config.QRELS_PATH)
gold = qr.groupby("query_id")["article_id"].apply(set).to_dict()
DOC = {"encoding": "utf-8-sig"}
nghin = lambda x: f"{x:,}".replace(",", ".")
print(f"Xong sau {time.time() - t0:.0f} giây: {nghin(len(tro_ly.tieu_de))} điều luật, "
      f"{nghin(len(tro_ly.de.chunk_ids))} đoạn")

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Xong sau 14 giây: 61.425 điều luật, 154.176 đoạn


## 3. Dữ liệu: câu hỏi trùng giữa train và test

Bộ gốc không có val nên nhóm cắt val ra từ train. 24 câu nằm ở cả train và test đều
mang nhãn khác nhau ở hai tệp.

In [13]:
display(pd.read_csv(config.AUDIT_DIR / "split_distribution.csv", **DOC))
xung = pd.read_csv(config.AUDIT_DIR / "conflicting_gold.csv", **DOC)
print(f"Câu trùng mã giữa train và test: {len(xung)}, "
      f"nhãn khác nhau: {(xung['giong_nhau'].astype(str).str.lower() == 'false').sum()}")
display(xung[["text", "gold_train", "gold_test"]].head(3))

ids = {s: set(g["query_id"]) for s, g in qr.groupby("split")}
print("Sau khi chia lại, số câu chung giữa các tập:",
      len(ids["train"] & ids["val"]), len(ids["train"] & ids["test"]), len(ids["val"] & ids["test"]))

,split,so_cau_hoi,so_cap_qrel,gold_trung_binh
0,test,788,793,1.006
1,train,1925,1986,1.032
2,val,481,492,1.023


Câu trùng mã giữa train và test: 24, nhãn khác nhau: 24


,text,gold_train,gold_test
0,"Bị thương tật 25%, cơ quan nhà nước có khởi tố không?",12/2017/qh14+1,101/2015/qh13+155
1,Tiêu chí xác định thương nhân áp dụng chế độ Luồng Đỏ được quy định như thế ...,31/2018/nđ-cp+29,15/2018/tt-bct+6
2,Đi trường giáo dưỡng được gọi điện thoại về nhà 5 phút/lần được quy định như...,43/2015/tt-bca+2,43/2015/tt-bca+12


Sau khi chia lại, số câu chung giữa các tập: 0 0 0


## 4. Một câu hỏi qua ba tầng

BM25, tầng ngữ nghĩa và hợp nhất, mỗi tầng lấy 3 điều đầu.

In [14]:
def ba_tang(cau, k=3):
    bm = tro_ly.bm.search_tokens(textnorm.tokens(cau), K)
    de = tro_ly.de.search(tro_ly.de.encode_queries([cau]), top_k=K,
                          pooling=tro_ly.ts["dense"]["gop_doan"], chunk_top=config.CHUNK_TOP)[0]
    hop = fusion.weighted_sum(bm, de, tro_ly.ts["weighted"]["alpha"], K)
    dong = []
    for ten, ds in (("BM25", bm), ("Ngữ nghĩa", de), ("Hợp nhất", hop)):
        for i, (aid, diem) in enumerate(ds[:k], 1):
            dong.append((ten, i, aid, tro_ly.tieu_de[aid][:70], round(diem, 3)))
    return pd.DataFrame(dong, columns=["tầng", "hạng", "điều luật", "tiêu đề", "điểm"])

cau = "Người lao động nghỉ việc có được trả lương những ngày chưa nghỉ phép không?"
display(ba_tang(cau))

,tầng,hạng,điều luật,tiêu đề,điểm
0,BM25,1,58/2010/qh12+13,Điều 13. Quyền của viên chức về nghỉ ngơi,17.557
1,BM25,2,01/2018/tt-bnv+23,Điều 23. Chính sách đối với giảng viên,16.340
2,BM25,3,105/2016/ttlt-bqp-bca-blđtbxh+3,Điều 3. Điều kiện hưởng chế độ ốm đau,15.191
3,Ngữ nghĩa,1,19/2014/tt-blđtbxh+18,"Điều 18. Tiền lương tính trả cho ngày nghỉ hằng năm, nghỉ lễ, tết",0.611
4,Ngữ nghĩa,2,27/2014/nđ-cp+16,Điều 16. Tiền lương ngừng việc,0.590
5,Ngữ nghĩa,3,19/2014/tt-blđtbxh+9,Điều 9. Nghĩa vụ của người lao động và người sử dụng lao động khi đơn,0.578
6,Hợp nhất,1,19/2014/tt-blđtbxh+18,"Điều 18. Tiền lương tính trả cho ngày nghỉ hằng năm, nghỉ lễ, tết",0.750
7,Hợp nhất,2,27/2014/nđ-cp+16,Điều 16. Tiền lương ngừng việc,0.673
8,Hợp nhất,3,19/2014/tt-blđtbxh+9,Điều 9. Nghĩa vụ của người lao động và người sử dụng lao động khi đơn,0.627


## 5. Câu hỏi gõ không dấu

Bốn câu thật trong tập test. Cột số là hạng của điều luật đúng trong kết quả hợp nhất,
trống nghĩa là không có trong top-100.

In [15]:
from vlr import diacritics

def hang_dung(cau, dung):
    bm = tro_ly.bm.search_tokens(textnorm.tokens(cau), K)
    de = tro_ly.de.search(tro_ly.de.encode_queries([cau]), top_k=K,
                          pooling=tro_ly.ts["dense"]["gop_doan"], chunk_top=config.CHUNK_TOP)[0]
    hop = fusion.weighted_sum(bm, de, tro_ly.ts["weighted"]["alpha"], K)
    return next((i for i, (a, _) in enumerate(hop, 1) if a in dung), None)

vi_du = ["01191d25556225e3961e0b4a76567c07", "03d587dd91c8f4c57fc862e413e3b461",
         "07aea7f0de54d5a441e6bf38a8e20039", "0825a491aab92827292a50ea01cfe61a"]
dong = []
for q in vi_du:
    goc = qs[q]
    khong = diacritics.bo_dau(goc)
    ph = tro_ly.ph.phuc_hoi(khong)
    dong.append((khong, ph, hang_dung(goc, gold[q]), hang_dung(khong, gold[q]), hang_dung(ph, gold[q])))
bang = pd.DataFrame(dong, columns=["gõ không dấu", "sau phục hồi", "câu gốc", "không dấu", "phục hồi"])
display(bang.astype({c: "Int64" for c in ["câu gốc", "không dấu", "phục hồi"]}))

,gõ không dấu,sau phục hồi,câu gốc,không dấu,phục hồi
0,Toa gia dinh co tham quyen xet xu cac vu an hinh su nao?,Tòa gia đình có thẩm quyền xét xử các vụ án hình sự nào?,1,<NA>,1
1,Bat giam dai bieu quoc hoi co phai co su dong y cua Quoc hoi khong?,Bắt giam đại biểu quốc hội có phải có sự đồng ý của Quốc hội không?,1,<NA>,1
2,Nhan vien thiet bi duoc xep luong the nao?,Nhân viên thiết bị được xếp lương thế nào?,1,<NA>,1
3,Cong ty dau tu chung khoan duoc hieu the nao?,Công ty đầu tư chứng khoán được hiểu thế nào?,1,<NA>,1


Phục hồi dấu cho vài kiểu gõ: không dấu, dấu một nửa, viết hoa, và một câu có từ nói thường mà luật không dùng.

In [16]:
for s in ["di xe may khong doi mu bao hiem bi phat bao nhieu tien",
          "đi xe may không đội mu bao hiem",
          "NGHI VIEC KHONG BAO TRUOC CO DUOC TRO CAP KHONG",
          "tai xe uong ruou bi phat the nao"]:
    t0 = time.perf_counter()
    kq = tro_ly.ph.phuc_hoi(s)
    print(f"{s}\n  -> {kq}   ({1000 * (time.perf_counter() - t0):.1f} ms)\n")

di xe may khong doi mu bao hiem bi phat bao nhieu tien
  -> đi xe máy không đội mũ bảo hiểm bị phạt bao nhiêu tiền   (0.5 ms)

đi xe may không đội mu bao hiem
  -> đi xe máy không đội mũ bảo hiểm   (0.2 ms)

NGHI VIEC KHONG BAO TRUOC CO DUOC TRO CAP KHONG
  -> NGHỈ VIỆC KHÔNG BÁO TRƯỚC CÓ ĐƯỢC TRỢ CẤP KHÔNG   (0.6 ms)

tai xe uong ruou bi phat the nao
  -> tải xe uống rượu bị phạt thế nào   (0.3 ms)



Kết quả trên cả 788 câu test, đọc từ `reports/eval/khong_dau.csv`:

In [17]:
kq = pd.read_csv(config.EVAL_DIR / "khong_dau.csv", **DOC)
kq = kq[kq["he_thong"] == "Hợp nhất"][["dieu_kien", "cach_xu_ly", "recall@1", "recall@10"]]
display(kq.reset_index(drop=True))

,dieu_kien,cach_xu_ly,recall@1,recall@10
0,có dấu,giữ nguyên,0.7824,0.9772
1,không dấu,giữ nguyên,0.0343,0.1447
2,không dấu,phục hồi dấu,0.7722,0.9670
3,nửa dấu,giữ nguyên,0.4524,0.7481
4,nửa dấu,phục hồi dấu,0.7786,0.9721
5,có dấu,phục hồi dấu,0.7824,0.9772


## 6. Trợ lý tra cứu

Phục hồi dấu, tìm bằng hệ hợp nhất, rồi trích nguyên văn khoản luật. Không sinh chữ.

In [18]:
print(tro_ly.tra_loi("Nguoi lao dong nghi viec co duoc tra luong nhung ngay chua nghi phep khong?"))

Bạn hỏi: Nguoi lao dong nghi viec co duoc tra luong nhung ngay chua nghi phep khong?
(Đã phục hồi dấu thành: Người lao động nghỉ việc có được trả lương những ngày chưa nghỉ phép không?)

Điều luật phù hợp nhất: Điều 18. Tiền lương tính trả cho ngày nghỉ hằng năm, nghỉ lễ, tết  [19/2014/tt-blđtbxh+18]
Nội dung liên quan: “3. Người lao động do chấm dứt hợp đồng lao động hoặc vì lý do khác mà chưa nghỉ hàng năm hoặc chưa nghỉ hết số ngày nghỉ hàng năm theo quy định thì được người sử dụng lao động thanh toán tiền lương những ngày người lao động chưa nghỉ. Tiền lương làm căn cứ tính trả cho những ngày người lao động chưa nghỉ là tiền lương tháng ghi trên hợp đồng lao động bình quân 6 tháng trước khi chấm dứt hợp đồng lao động hoặc trước khi tính trả cho người lao động, chia cho số ngày làm việc bình thường trong tháng theo quy định của pháp luật mà hai bên xác định nhưng tối đa không quá 26 ngày, nhân với số ngày ...”
Từ khớp: trả (5.0992), lao_động (4.9747), ngày (2.5386), người (2.0654)



Câu nằm ngoài kho luật: trợ lý vẫn trả về một điều, nhưng in kèm cảnh báo.

In [19]:
print(tro_ly.tra_loi("hom nay troi dep khong"))

Bạn hỏi: hom nay troi dep khong
(Đã phục hồi dấu thành: hôm nay trời đẹp không)
(Cảnh báo: không từ nào trong câu hỏi khớp điều luật bên dưới, nhiều khả năng câu hỏi nằm ngoài phạm vi kho)

Điều luật phù hợp nhất: Điều 4. Giải thích từ ngữ  [03/2020/qđ-ttg+4]
Nội dung liên quan: “14. Lốc là luồng gió xoáy có sức gió mạnh tương đương với sức gió của bão nhưng được hình thành và tan trong thời gian ngắn, phạm vi hoạt động hẹp từ vài km2 đến vài chục km2. 15. Sét là hiện tượng phóng điện trong đám mây, giữa các đám mây với nhau hoặc giữa đám mây với mặt đất. 16. Mưa đá là mưa dưới dạng cục băng hoặc hạt băng có kích thước, hình dạng khác nhau, xảy ra trong thời gian ngắn, kèm theo mưa rào, đôi khi có gió mạnh. 17. Mưa lớn là hiện tượng mưa với tổng lượng mưa đạt trên 50 mm trong 24 giờ, trong đó mưa với tổng lượng mưa từ trên 50 mm đến 100 mm trong 24 giờ là mưa to, mưa ...”
Từ khớp: (không từ nào khớp)

Tham khảo thêm:
  - Điều 1. Sửa đổi, bổ sung một số điều của Nghị định số 38/2016/NĐ-

Gõ câu hỏi khác vào đây rồi chạy lại ô:

In [20]:
cau_hoi = "do tuoi toi thieu duoc ket hon la bao nhieu"
print(tro_ly.tra_loi(cau_hoi))

Bạn hỏi: do tuoi toi thieu duoc ket hon la bao nhieu
(Đã phục hồi dấu thành: độ tuổi tối thiểu được kết hôn là bao nhiêu)

Điều luật phù hợp nhất: Điều 8. Điều kiện kết hôn  [52/2014/qh13+8]
Nội dung liên quan: “1. Nam, nữ kết hôn với nhau phải tuân theo các điều kiện sau đây: a) Nam từ đủ 20 tuổi trở lên, nữ từ đủ 18 tuổi trở lên; b) Việc kết hôn do nam và nữ tự nguyện quyết định; c) Không bị mất năng lực hành vi dân sự; d) Việc kết hôn không thuộc một trong các trường hợp cấm kết hôn theo quy định tại các điểm a, b, c và d khoản 2 Điều 5 của Luật này. 2. Nhà nước không thừa nhận hôn nhân giữa những người cùng giới tính.”
Từ khớp: kết_hôn (11.7324), tuổi (5.4153)

Tham khảo thêm:
  - Điều 3. Giải thích từ ngữ  [52/2014/qh13+3]
  - Điều 2. Căn cứ hủy việc kết hôn trái pháp luật  [01/2016/ttlt-tandtc-vksndtc-btp+2]

Trợ lý chỉ trích nguyên văn điều luật, không diễn giải. Hãy đọc toàn văn điều trước khi viện dẫn.
